# Gold v2 — dev 세트 만들기 (agent 설계·선택용)

gold v2가 Aligned Core의 결함 이미지를 전부 가져갔기 때문에, dev 세트는 **supervised train**에서 뽑습니다.
논문 수치는 gold v2로만 내고, dev는 agent 구성을 고르는 데만 씁니다.

- 구성 (seed 42): 자산별 결함 25장(vari-grip은 새 둥지 20 + rust 20) + 정상 25장. polymer 결함(corrosão)은 rust와 합치지 않는 규칙 때문에 제외 → polymer는 정상만
- 제외: unsupervised test에 있는 이미지 / 정상 reference 후보(unsupervised train 정상)와 같은 이미지 / gold v2 이미지 / 중복
- 문항: gold v2와 같은 문구·선택지의 Q1(정상/이상)·Q2(결함 종류). Q3는 없음(위치 라벨 없음)
- 정상 reference 2종: gold와 같은 규칙의 **고정 1장(자산별)** + **비슷한 정상 이미지(DINOv2 cosine top-1)**

런타임: **GPU 권장(T4나 L4)** — 정상 후보 약 9,000장을 DINOv2로 인코딩합니다(GPU 수 분, CPU면 1시간 이상). 인코딩 결과는 Drive에 저장돼 끊겨도 이어집니다.

결과: `MyDrive/lvlm_insplad_qa/gold_v2_dev/` → GitHub `results/latest/gold_v2_dev/` (JSON·CSV만, 이미지는 올리지 않음)

In [ ]:
REPO = "arnold6444/lvlm_insplad"
REPO_REF = "main"
GOLD_QA_FOLDER = "lvlm_insplad_qa/gold_v2_final"
DEV_FOLDER = "lvlm_insplad_qa/gold_v2_dev"
DEFECT_PER_CLASS = 25
NORMAL_PER_ASSET = 25
VARI_GRIP_PER_DEFECT = 20
PUBLISH = True

UNSUP_ZIP_NAME = "unsupervised_anomaly_detection.zip"
UNSUP_ZIP_SIZE = 1_170_967_719
UNSUP_ZIP_PATH = ""   # 비우면 자동 검색
SUP_ZIP_NAME = "supervised_fault_classification.zip"
SUP_ZIP_SIZE = 874_753_167
SUP_ZIP_PATH = ""     # 비우면 자동 검색

In [ ]:
# Drive 연결 + repo clone + 패키지
import base64, json, os, shutil, subprocess, sys
from pathlib import Path
from google.colab import drive, userdata

drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = Path("/content/drive/MyDrive")
GOLD_QA_DIR = DRIVE_ROOT / GOLD_QA_FOLDER
DEV_DIR = DRIVE_ROOT / DEV_FOLDER
DEV_DIR.mkdir(parents=True, exist_ok=True)

REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

def run(cmd, *, env=None, check=True):
    """Run a command and stream its output into this cell (Colab hides plain subprocess output)."""
    cmd = [str(x) for x in cmd]
    print("+", " ".join(cmd[:12]) + (" ..." if len(cmd) > 12 else ""), flush=True)
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail = []
    for line in p.stdout:
        tail = (tail + [line])[-60:]
        print(line, end="", flush=True)
    p.wait()
    if check and p.returncode:
        raise RuntimeError(f"command failed with exit code {p.returncode} (마지막 출력 위 참고)")
    return p.returncode

github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab Secret GITHUB_TOKEN이 필요합니다.")
os.environ["GITHUB_TOKEN"] = github_token
auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env = os.environ.copy()
git_env.update({"GIT_CONFIG_COUNT":"1", "GIT_CONFIG_KEY_0":"http.extraHeader",
                "GIT_CONFIG_VALUE_0":f"AUTHORIZATION: basic {auth}", "GIT_TERMINAL_PROMPT":"0"})
run(["git","clone","--depth","1","--branch",REPO_REF,f"https://github.com/{REPO}.git",REPO_DIR], env=git_env)
del auth, git_env
run([sys.executable,"-m","pip","install","-q","transformers==5.14.0","accelerate==1.14.0","pillow>=10","pandas>=2.2"])
sys.path.insert(0, str(REPO_DIR / "src"))
sys.path.insert(0, str(REPO_DIR / "scripts"))
import pandas as pd
import torch
print("repo:", subprocess.check_output(["git","rev-parse","HEAD"], cwd=REPO_DIR, text=True).strip())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU로 진행 — 인코딩이 오래 걸림)")

In [ ]:
def find_zip(filename, expected_size, override=""):
    candidates = [Path(override)] if override else []
    candidates += [DRIVE_ROOT / filename, DRIVE_ROOT / "lvlm_insplad_data" / filename]
    for p in candidates:
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    print(f"{filename}: Drive 전체 검색 중 (몇 분 걸릴 수 있음)", flush=True)
    for p in DRIVE_ROOT.rglob(filename):
        if p.is_file() and p.stat().st_size == expected_size:
            return p
    raise FileNotFoundError(f"{filename} ({expected_size} bytes) not found")

SUP_ZIP = find_zip(SUP_ZIP_NAME, SUP_ZIP_SIZE, SUP_ZIP_PATH)
UNSUP_ZIP = find_zip(UNSUP_ZIP_NAME, UNSUP_ZIP_SIZE, UNSUP_ZIP_PATH)
print("SUP:", SUP_ZIP); print("UNSUP:", UNSUP_ZIP)

In [ ]:
# dev 세트 생성 (이미지 선정 + 고정 reference + DINOv2 비슷한 정상 이미지 검색)
run([sys.executable, REPO_DIR / "scripts/build_gold_v2_dev_set.py",
     "--sup-zip", SUP_ZIP, "--unsup-zip", UNSUP_ZIP, "--gold-qa-dir", GOLD_QA_DIR, "--out-dir", DEV_DIR,
     "--defect-per-class", str(DEFECT_PER_CLASS), "--normal-per-asset", str(NORMAL_PER_ASSET),
     "--vari-grip-per-defect", str(VARI_GRIP_PER_DEFECT)])
summary = json.loads((DEV_DIR / "build_summary.json").read_text(encoding="utf-8"))
print(json.dumps({k: summary[k] for k in ("images","questions","by_asset_defect","quality_status")}, ensure_ascii=False, indent=1))
print("제외:", summary["selection"]["excluded"])
print("부족:", summary["selection"]["shortfall"] or "없음")
print("검색:", summary["retrieval"])

In [ ]:
# 확인 1: 고정 reference가 gold v2와 같은 이미지인지 (같은 규칙이면 같아야 함)
gold_rows = [json.loads(x) for f in sorted(GOLD_QA_DIR.glob("master_qa_*.jsonl"))
             for x in f.read_text(encoding="utf-8").splitlines() if x.strip()]
gold_ref = {}
for r in gold_rows:
    gold_ref.setdefault(r["asset_category"], set()).add(r["normal_reference_image"])
ok = True
for asset, ref in summary["fixed_reference_by_asset"].items():
    same = gold_ref.get(asset) == {ref}
    ok &= same
    print(("일치" if same else "다름"), asset, ref, "" if same else f"(gold: {gold_ref.get(asset)})")
print("고정 reference 검증:", "통과" if ok else "불일치 — 알려주세요")

In [ ]:
# 확인 2: 고정 reference vs 비슷한 정상 이미지 예시 (자산별 1장씩)
import io
from zipfile import ZipFile
from PIL import Image
import matplotlib.pyplot as plt

dev = pd.read_csv(DEV_DIR / "dev_images.csv")
examples = dev.groupby("asset_category").head(1)
sup_z, unsup_z = ZipFile(SUP_ZIP), ZipFile(UNSUP_ZIP)
def load(z, member):
    return Image.open(io.BytesIO(z.read(member))).convert("RGB")
fig, axes = plt.subplots(len(examples), 3, figsize=(9, 3 * len(examples)))
for ax_row, (_, r) in zip(axes, examples.iterrows()):
    for ax, (title, img) in zip(ax_row, [("target: " + r.defect_gt, load(sup_z, r.image)),
                                         ("fixed ref", load(unsup_z, r.normal_reference_image)),
                                         (f"retrieved (cos {r.retrieval_cosine:.2f})", load(unsup_z, r.retrieved_reference_image))]):
        ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# GitHub publish (JSON·CSV만)
if PUBLISH and summary.get("quality_status") == "FROZEN":
    run([sys.executable, REPO_DIR / "scripts/publish_results_to_github.py", "--out-dir", DEV_DIR,
         "--repo", REPO, "--branch", REPO_REF, "--prefix", "gold_v2_dev",
         "--files", "build_summary.json", "dev_images.csv", summary["file"]], env=os.environ.copy())
    print("GitHub: results/latest/gold_v2_dev/")
else:
    print("publish 안 함:", summary.get("quality_status"))